# Band 7 ablation with capped failure feedback

Run on the same Colab GPU type and numerical runtime as the archived full-map run. The notebook loads the **Band 6 Round 3 adapter**, then refits once using the archived cumulative replay through **Band 7 Round 1**. There is no new experience collection for the fit.

The refit starts with the same minibatch draws as the saved control. It replaces only failure draws above the cap, using successful pairs from the same goal wherever possible. With the default **15% cap**, the 32-pair minibatches get at most four or five failure draws each and at most 15% across the run. The notebook reports the actual draws and replacements. Success pair labels, the loss scale and the optimizer schedule are reused.

The fitted adapter is evaluated on the **66 archived Band 7 Round 2 starts**. The saved Round 2 outcomes supply the paired control. These evaluation rollouts supply no training pairs. The figures overlay the realized trajectories on the two planner costs for the actual collection goals.

Set `RESUME_RUN_ID` to the printed ID to reuse a completed fit and any finished evaluation cases. The original run is read from Drive and this ablation writes to a separate folder.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
RUNS=DRIVE/'datasets/curriculum_cost_adapter'
SOURCE_RUN=RUNS/'experience_mpc_failure_feedback_full_square_v1/20261109T072213743530Z'
FAILURE_FRACTION=.15
PLOT_GOALS=[1,4,5]
RESUME_RUN_ID=None
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=RUNS/'experience_failure_cap_v1'/RUN_ID
if not (SOURCE_RUN/'band_7_round_2/complete.json').is_file():
    raise FileNotFoundError(f'Missing archived Band 7 Round 2 results at {SOURCE_RUN}.')
REPO=Path('/content/i-jepa-wm-experience-failure-cap')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='c85e9da1190ea9ba366ab63784d221025d2590d9'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-square-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Run ID:',RUN_ID)
print('New experience curriculum results:',OUTPUT)

## Load the pinned code and frozen models

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

## Refit once and evaluate the fixed starts

The replay loader prints progress as it reads the archived tensors. It allocates the cumulative tensor once and copies the saved observations into it.

In [ ]:
started=time.monotonic()
shell([sys.executable,'-m','src.mylonas_ijepa_wm.experience_failure_cap',
       '--source-run',SOURCE_RUN,'--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
       '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
       '--fraction',FAILURE_FRACTION],cwd=REPO)
print('Minutes:',round((time.monotonic()-started)/60,2))

## Compare sampled replay and paired outcomes

Previous-band entry and doorway crossing are separate. An episode can enter the inner square without going through the opening. A win is a case solved only by the capped adapter. A loss is a case solved only by the archived control.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Image, display

report=json.loads((OUTPUT/'comparison.json').read_text())
sampling=report['sampling']
display(pd.DataFrame([
    dict(policy='Archived control',failure_draws=sampling['original_failure_draws']),
    dict(policy='Capped refit',failure_draws=sampling['capped_failure_draws']),
]).assign(total_draws=sampling['total_draws'],
          failure_pct=lambda table:100*table.failure_draws/table.total_draws).round(2))
print(f"Replaced {sampling['replacements']} draws. "
      f"{sampling.get('goal_changes',0)} replacements used a different goal.")

rows=[]
for row in report['cases']:
    case=row['case']
    old,new=row['original_success'],row['capped_success']
    rows.append(dict(route=case['route'],cases=1,control=int(old),capped=int(new),
        wins=int(new and not old),losses=int(old and not new),
        control_crossed=int(row['original_crossed']),capped_crossed=int(row['capped_crossed']),
        control_entry_by_5=int(row['original']['entered_by_five_actions']),
        capped_entry_by_5=int(row['capped']['entered_by_five_actions']),
        control_entry_ever=int(row['original']['first_entry_step'] is not None),
        capped_entry_ever=int(row['capped']['first_entry_step'] is not None)))
table=pd.DataFrame(rows).groupby('route').sum()
table.loc['all']=table.sum()
display(table)
table.to_csv(OUTPUT/'paired_summary.csv')

## Overlay the matched paths on the two planner costs

The panels share an absolute latent-MSE scale for each goal. Green traces successful rollouts. Red traces the first five actions of failures, and dashed red shows their start-to-end displacement. The amber diamond marks the first macro endpoint. Amber squares mark the Band 7 boundaries at half-widths 36 and 42.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

axis=report['axis']
shape=(len(axis),len(axis))
plot_dir=OUTPUT/'overlays'
plot_dir.mkdir(exist_ok=True)
for i in PLOT_GOALS:
    cases=[row for row in report['cases'] if row['case']['goal_index']==i]
    if not cases:
        continue
    gx,gy=cases[0]['case']['goal']
    fields=[np.asarray(report[key][i],dtype=float).reshape(shape) for key in ('control_grid','capped_grid')]
    vmax=max(float(np.nanmax(field)) for field in fields)
    fig,axes=plt.subplots(1,2,figsize=(10,5.3),layout='constrained')
    for ax,field,key,label in zip(axes,fields,('original','capped'),('Archived control','Capped refit')):
        mesh=ax.pcolormesh(axis,axis,field,shading='nearest',cmap='viridis',vmin=0,vmax=vmax)
        for y,height in ((4,22),(34,26)):
            ax.add_patch(Rectangle((29,y),6,height,facecolor='#555',alpha=.75,zorder=2))
        for radius,style in ((36,'-'),(42,'--')):
            ax.add_patch(Rectangle((gx-radius,gy-radius),2*radius,2*radius,
                fill=False,edgecolor='#ffab39',linestyle=style,linewidth=1.25,zorder=3))
        for row in sorted(cases,key=lambda r:r[key+'_success']):
            xy=np.asarray(row[key]['positions'])
            success=row[key+'_success']
            color='#64f39a' if success else '#ff727a'
            ax.plot(*(xy if success else xy[:6]).T,color=color,alpha=.88,linewidth=1.55,zorder=4)
            if not success:
                ax.plot(*xy[[0,-1]].T,color=color,linestyle='--',alpha=.55,zorder=4)
            ax.scatter(*xy[min(5,len(xy)-1)],s=28,marker='D',c='#ffc341',edgecolor='#252526',zorder=5)
            ax.scatter(*xy[0],s=40,facecolors='none',edgecolors=color,zorder=6)
            ax.scatter(*xy[-1],s=40,color=color,marker='x',zorder=6)
        ax.scatter(gx,gy,marker='*',s=120,c='black',edgecolor='white',zorder=7)
        wins=sum(row[key+'_success'] for row in cases)
        crossed=sum(row[key+'_crossed'] for row in cases)
        ax.set(title=f'{label} · {wins}/{len(cases)} success · {crossed} crossed',
               xlim=(4,60),ylim=(4,60),aspect='equal',xlabel='Ball x',ylabel='Ball y')
    fig.colorbar(mesh,ax=axes.tolist(),shrink=.68,label='Latent MSE')
    fig.suptitle(f'Band 7 Round 2 · Collection goal {i} ({gx:.2f}, {gy:.2f})')
    fig.legend(handles=[
        Line2D([],[],color='#1b984e',label='Success'),
        Line2D([],[],color='#c44048',label='Failed first five'),
        Line2D([],[],color='#c44048',ls='--',label='Failed start to end'),
        Line2D([],[],marker='D',mfc='#ffc341',mec='#252526',ls='none',label='First macro endpoint'),
    ],loc='upper center',bbox_to_anchor=(.5,-.005),ncol=4,frameon=False)
    path=plot_dir/f'goal_{i}_paired.png'
    fig.savefig(path,dpi=140,bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=str(path)))